# 1. Introdução aos Widgets do Databricks

## Visão Geral
Os widgets do Databricks permitem a criação de elementos interativos dentro dos notebooks que facilitam a passagem de parâmetros entre diferentes execuções. Esses widgets são amplamente utilizados para parametrizar notebooks, tornando-os mais dinâmicos e flexíveis para diferentes cenários, como separar ambiente de desenvolvimento e produção ou definir parâmetros para modelos de machine learning.

## Tipos de Widgets
O Databricks oferece diferentes tipos de widgets que podem ser utilizados de acordo com a necessidade:
- **Texto**: Um campo de texto livre para entrada de valores.
- **Dropdown**: Um menu suspenso que permite a seleção de um valor de uma lista pré-definida.
- **Combobox**: Combinação de um campo de texto com opções de auto-completar de uma lista.
- **Multiselect**: Permite selecionar múltiplos valores de uma lista.

Esses widgets são fundamentais para criar notebooks interativos que se adaptam a diferentes inputs.

## Sintaxe Básica
Os widgets são criados e manipulados através da função `dbutils.widgets`. Vamos ver alguns exemplos de como criar e utilizar esses widgets.




### Criando um Widget de Texto
Para criar um widget de texto, use o seguinte comando:

In [0]:
# Text Widget: A free-text input field
dbutils.widgets.text("input_text", "valor_padrao", "Widget de Texto: Insira um valor")

# Retrieve and display the value of the text widget
text_value = dbutils.widgets.get("input_text")
print("Valor do Widget de Texto:", text_value)

### Criando um Widget de Menu Suspenso

In [0]:
# Dropdown Widget: A drop-down menu with pre-defined options
dbutils.widgets.dropdown("dropdown_widget", "Opção1", ["Opção1", "Opção2", "Opção3"], "Widget de menu suspenso: Selecione uma opção")

# Retrieve and display the value of the menu suspenso widget
dropdown_value = dbutils.widgets.get("dropdown_widget")
print("Valor do Widget de menu suspenso:", dropdown_value)


### Combobox Widget

In [0]:
# Combobox Widget: A combination of a text field with auto-complete options
dbutils.widgets.combobox("combobox_widget", "Opção1", ["Opção1", "Opção2", "Opção3"], "Widget de Combobox: Digite ou selecione uma opção")

# Retrieve and display the value of the combobox widget
combobox_value = dbutils.widgets.get("combobox_widget")
print("Valor do Widget de Combobox:", combobox_value)

### Widget de Seleção Múltipla

In [0]:
# Multiselect Widget: Allows selection of multiple values from a pre-defined list
dbutils.widgets.multiselect("multiselect_widget", "Opção1", ["Opção1", "Opção2", "Opção3"], "Widget de Multiseleção: Selecione uma ou mais opções")

# Retrieve and display the values of the multiselect widget
multiselect_values = dbutils.widgets.get("multiselect_widget")
print("Valores do Widget de Multiseleção:", multiselect_values)


## Imports

In [0]:
import os
import warnings
import yaml

dbutils.widgets.text("conf_filepath", 'conf/')

### Função de Leitura de YAML

In [0]:
def read_yaml_file(file_path):
    """
    Reads a YAML file and returns its content as a Python dictionary.

    Parameters:
        file_path (str): The path to the YAML file.

    Returns:
        dict: Content of the YAML file.

    Raises:
        Exception: If there is an error loading or parsing the YAML file.
    """
    try:
        with open(file_path, 'r') as file:
            return yaml.safe_load(file)  # Parse and load YAML content
    except FileNotFoundError:
        raise Exception(f"File not found: {file_path}")
    except yaml.YAMLError as e:
        raise Exception(f"Error processing the YAML file: {file_path}. {e}")

def load_configuration(filepath):
    """
    Determines which configuration file (dev or prod) to load based on file existence.

    Parameters:
        filepath (str): Base directory path where configuration files are located.

    Returns:
        dict: Content of the selected YAML configuration file.

    Warnings:
        Emits a warning if the dev configuration file is missing and uses prod as a fallback.
    """
    # Paths to the dev and prod configuration files
    dev_file = os.path.join(filepath, "dev", "environment.yml")
    prod_file = os.path.join(filepath, "prod", "environment.yml")

    # Choose which file to load: prefer dev, fallback to prod
    file_to_load = dev_file if os.path.isfile(dev_file) else prod_file

    if file_to_load == prod_file:
        warnings.warn(f"Dev configuration not found. Using prod configuration: {prod_file}.", UserWarning)

    # Load and return the content of the selected file
    return read_yaml_file(file_to_load)

### Função de Criação de Widgets

In [0]:
def create_widgets(config):
    """
    Creates Databricks widgets based on variables defined in the configuration.

    Parameters:
        config (dict): Configuration loaded from the YAML file.

    Raises:
        Exception: If the configuration does not contain the 'variables' key.
    """
    if 'variables' not in config:
        raise Exception("The configuration does not contain the key 'variables'.")

    for key, value in config['variables'].items():
        # Get the default value for each variable, or an empty string if not specified
        default = value.get("default", "")
        # Create a Databricks widget for each variable with its default value
        dbutils.widgets.text(str(key), str(default))


In [0]:
def set_spark_config_from_widgets(config):
    """
    Sets Spark configurations based on the values from Databricks widgets.

    Parameters:
        config (dict): Configuration loaded from the YAML file.

    Returns:
        dict: A dictionary of widget variable names and their corresponding values.

    Raises:
        Exception: If a widget value is empty or None.
    """
    vars_config = {}
    for key in config['variables']:
        value = dbutils.widgets.get(key)
        if not value:
            raise Exception(f"The value for the variable '{key}' cannot be empty.")
        vars_config[key] = value
        # Set the variable as a Spark configuration
        spark.conf.set(f'vars.{key}', value)


In [0]:
def _read_yaml_file(file_path):

    with open(file_path, 'r') as file:
        try: 
            data = yaml.safe_load(file)
            return data
        
        except yaml.YAMLError as e:
            print(f"Failed to load {file_path}.", e)

def load_conf(conf_filepath: str):
    "Resolve environment and load default values."
    dev_filepath = os.path.join(conf_filepath, "dev", "environment.yml")
    prod_filepath = os.path.join(conf_filepath, "prod", "environment.yml")

    if os.path.isfile(dev_filepath):
        return _read_yaml_file(dev_filepath)

    warnings.warn("Dev configuration not found. Using prod configuration.", UserWarning)
    return _read_yaml_file(prod_filepath)

In [0]:
environment_variables = load_conf(dbutils.widgets.get('conf_filepath')) 

In [0]:
# set default widget values based on filepath
for key, value in environment_variables['variables'].items():
    default_value=value.get("default", "") 
    dbutils.widgets.text(str(key), str(default_value))

In [0]:
vars = {variable: dbutils.widgets.get(variable) for variable in environment_variables["variables"].keys()}

for key, value in vars.items():
    if value is None:
        raise Exception(f'{key} cant be none')
    spark.conf.set(f'vars.{key}', value)